In [ ]:
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/Talia-commits-creator/plp-da-a3-duka-sales-and-agent-cleanup.git"
PROJECT_DIR = Path("/content/plp-da-a3-duka-sales-and-agent-cleanup")

# Google Colab opens the notebook separately from the other repository files.
# Clone the public repository into the runtime the first time this cell runs.
if not PROJECT_DIR.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(PROJECT_DIR)],
        check=True
    )

required_files = ["generate_data.py", "cleaning_pipeline.py"]
missing_files = [name for name in required_files if not (PROJECT_DIR / name).is_file()]
if missing_files:
    raise FileNotFoundError(
        f"Project files are missing from {PROJECT_DIR}: {missing_files}. "
        "In Colab, delete the incomplete project folder from the Files pane "
        "and run this setup cell again."
    )

os.chdir(PROJECT_DIR)
print("Project folder:", PROJECT_DIR)
print("Current working directory:", Path.cwd())
print("Project files found:", ", ".join(required_files))

In [ ]:
import subprocess
import sys

generator = PROJECT_DIR / "generate_data.py"
subprocess.run([sys.executable, str(generator)], cwd=PROJECT_DIR, check=True)

expected_files = [
    "duka_sales.csv",
    "agents.csv",
    "agent_transactions.csv",
]
missing_files = [name for name in expected_files if not (PROJECT_DIR / name).is_file()]

if missing_files:
    raise FileNotFoundError(f"Generator did not create: {missing_files}")

print("All source datasets generated successfully.")
for name in expected_files:
    print(f"- {name}: {(PROJECT_DIR / name).stat().st_size:,} bytes")

In [ ]:
import pandas as pd
import numpy as np

sales = pd.read_csv(
    "duka_sales.csv",
    parse_dates=["order_date"]
)

print("FIRST FIVE ROWS")
display(sales.head())

print("LAST FIVE ROWS")
display(sales.tail())

print("DATASET SHAPE")
print(sales.shape)

print("DATA TYPES")
sales.info()

print("STATISTICAL SUMMARY")
display(sales.describe())

print("UNIQUE VALUES")
display(sales.nunique())

print("MISSING VALUES")
display(sales.isna().sum())

print("DATE RANGE")
print(sales["order_date"].min(), "to", sales["order_date"].max())

In [ ]:
print("BRANCH DISTRIBUTION")
display(sales["branch"].value_counts(normalize=True).mul(100).round(2))

print("CHANNEL DISTRIBUTION")
display(sales["channel"].value_counts(normalize=True).mul(100).round(2))

Nairobi accounts for the largest proportion of orders (37.6%), while Eldoret has the smallest share (11.7%). In-store purchases dominate the sales channels at 57.9%, followed by online purchases at 32.0% and phone orders at 10.1%. This suggests that physical stores are Duka's primary sales channel, although online sales also represent a substantial part of the business

In [ ]:
sales["revenue"] = sales["quantity"] * sales["unit_price"]

sales["month"] = sales["order_date"].dt.month_name()

sales["weekday"] = sales["order_date"].dt.day_name()

sales["is_weekend"] = sales["order_date"].dt.dayofweek >= 5

sales["price_band"] = np.where(
    sales["unit_price"] >= 28000,
    "Premium",
    "Value"
)

display(sales.head())

In [ ]:
branch_performance = sales.groupby("branch").agg(
    total_revenue=("revenue", "sum"),
    total_orders=("order_id", "count")
)

branch_performance = branch_performance.sort_values(
    by="total_revenue",
    ascending=False
)

display(branch_performance)

In [ ]:
print("Branch ranked by revenue:")
display(branch_performance["total_revenue"].sort_values(ascending=False))

print("Branch ranked by number of orders:")
display(branch_performance["total_orders"].sort_values(ascending=False))

Nairobi is the best-performing branch, generating the highest revenue of KES 26,106,100 from 376 orders. Eldoret is the lowest-performing branch, generating KES 8,844,200 from 117 orders. The revenue and order-volume rankings are identical, suggesting that branches with more orders also generate higher total revenue. Duka should investigate Nairobi's successful sales practices and identify ways to improve performance in Eldoret.

In [ ]:
product_performance = sales.groupby("product").agg(
    total_revenue=("revenue", "sum"),
    units_sold=("quantity", "sum")
)

print("TOP 3 PRODUCTS BY REVENUE")
display(
    product_performance
    .sort_values("total_revenue", ascending=False)
    .head(3)
)

print("TOP 3 PRODUCTS BY UNITS SOLD")
display(
    product_performance
    .sort_values("units_sold", ascending=False)
    .head(3)
)

Laptops generate the highest revenue (KES 41,535,000) despite selling only 639 units. Tablets have the highest sales volume, with 691 units sold, generating KES 19,348,000. Phones rank third in both revenue and units sold. This shows that high-priced products can generate more revenue even when fewer units are sold. Duka should maintain laptop availability while continuing to promote tablets due to their high sales volume.

In [ ]:
weekend_analysis = sales.groupby("is_weekend").agg(
    total_orders=("order_id", "count"),
    total_revenue=("revenue", "sum"),
    average_revenue=("revenue", "mean")
)

weekend_analysis.index = ["Weekdays", "Weekends"]

display(weekend_analysis.round(2))

Weekdays generate higher total revenue (KES 51,856,200) than weekends (KES 19,936,300), mainly because they account for more orders (723 compared with 277). However, weekends have a slightly higher average revenue per order (KES 71,972.20) than weekdays (KES 71,723.65). Duka could explore weekend promotions to increase order volume while maintaining the higher average order value.

In [ ]:
monthly_sales = sales.groupby("month").agg(
    total_revenue=("revenue", "sum"),
    total_orders=("order_id", "count")
)

# Arrange months in calendar order
month_order = [
    "January", "February", "March",
    "April", "May", "June"
]

monthly_sales = monthly_sales.reindex(month_order)

display(monthly_sales)

best_month = monthly_sales["total_revenue"].idxmax()
worst_month = monthly_sales["total_revenue"].idxmin()

best_revenue = monthly_sales["total_revenue"].max()
worst_revenue = monthly_sales["total_revenue"].min()

percentage_difference = (
    (best_revenue - worst_revenue) / worst_revenue
) * 100

print("Best month:", best_month)
print("Worst month:", worst_month)
print("Percentage by which the best month exceeds the worst month:",
      round(percentage_difference, 2), "%")

April was the best-performing month, generating KES 13,518,300 in revenue from 167 orders. June was the weakest month, generating KES 10,903,400 from 162 orders. April's revenue was 23.98% higher than June's. Duka should investigate the factors behind April's strong performance and consider strategies to improve sales during June.

In [ ]:
premium_online = sales[
    (sales["price_band"] == "Premium") &
    (sales["channel"] == "Online") &
    (sales["branch"].isin(["Nairobi", "Mombasa"]))
].copy()

print("Number of qualifying orders:", len(premium_online))

print("Total revenue:", premium_online["revenue"].sum())

display(premium_online.head())

In [ ]:
import os

os.makedirs("outputs", exist_ok=True)

premium_online.to_csv(
    "outputs/premium_online_coast_capital.csv",
    index=False
)

print("File exported successfully!")

A total of 83 orders met the filtering criteria, generating KES 10,218,000 in revenue. These orders involved Premium products purchased online from the Nairobi and Mombasa branches. This segment represents an opportunity for Duka to strengthen its online sales strategy and target customers interested in higher-priced products.

In [ ]:
import os
import pandas as pd

file_path = "outputs/premium_online_coast_capital.csv"

print("File exists:", os.path.exists(file_path))

exported = pd.read_csv(file_path)

print("Exported shape:", exported.shape)
print("Exported revenue:", exported["revenue"].sum())

Executive Summary
1. Duka Electronics recorded 1,000 orders across five branches between January and June 2026.
2. Nairobi was the best-performing branch, generating KES 26,106,100 from 376 orders.
3. Laptops generated the highest product revenue of KES 41,535,000, while tablets recorded the highest sales volume at 691 units.
4. April was the strongest month, generating KES 13,518,300, which was 23.98% higher than June, the weakest month.
5. A total of 83 Premium online orders from Nairobi and Mombasa generated KES 10,218,000 in revenue.

In [ ]:
sales.isnull().sum()

Data Quality Observation
The dataset contains 1,000 records and 12 columns, with zero missing values across all columns. This indicates that the dataset is complete and does not require missing-value treatment before analysis. The dataset covers five branches and three sales channels over the period January to June 2026.

In [ ]:
from pathlib import Path

outputs_dir = PROJECT_DIR / "outputs"
outputs_dir.mkdir(exist_ok=True)
sales.to_csv(outputs_dir / "duka_sales_enriched.csv", index=False)
print("Enriched sales data saved to:", outputs_dir / "duka_sales_enriched.csv")

In [ ]:
workbook_path = PROJECT_DIR / "duka_branch_summary.xlsx"

print("Workbook exists:", workbook_path.is_file())
if workbook_path.is_file():
    print("Workbook size:", f"{workbook_path.stat().st_size:,} bytes")
else:
    print("The workbook is not in the project root yet; it will be created/validated in a later stage.")

In [ ]:
import os

print("Current folder:", os.getcwd())
print("\nProject files:")

for item in os.listdir():
    print(item)


In [ ]:
import pandas as pd
import numpy as np

agents = pd.read_csv("agents.csv")
txns = pd.read_csv("agent_transactions.csv")

print("AGENT REGISTER")
display(agents.head())

print("TRANSACTION DATA")
display(txns.head())

In [ ]:
print("AGENT REGISTER SHAPE:", agents.shape)
print("TRANSACTION SHAPE:", txns.shape)

print("\nAGENT REGISTER INFORMATION")
agents.info()

print("\nTRANSACTION INFORMATION")
txns.info()

print("\nTRANSACTION SUMMARY")
display(txns.describe(include="all"))

In [ ]:
print("MISSING VALUES")
display(txns.isna().sum())

print("\nDUPLICATE ROWS")
print(txns.duplicated().sum())

print("\nDUPLICATE TRANSACTION IDs")
print(txns["txn_id"].duplicated().sum())

print("\nTRANSACTION TYPES")
display(txns["txn_type"].value_counts(dropna=False))

print("\nAGENT IDs")
display(txns["agent_id"].value_counts().head(10))

The transaction dataset contains 612 records and five columns, while the agent register contains 20 agents and three columns. The transaction dataset has 12 duplicate transaction IDs, 11 exact duplicate rows and 22 missing transaction amounts. Transaction types have five inconsistent representations that need standardization. The maximum transaction amount is KES 250,000, which requires outlier investigation. The initial inspection also indicates inconsistent date formatting and potentially unregistered agent references that require further validation.

In [ ]:
# 1. Check unusual transaction dates
print("DATE FORMATS")
display(txns["txn_date"].value_counts().head(15))

# 2. Check unregistered agents
unknown_agents = set(txns["agent_id"]) - set(agents["agent_id"])

print("\nUNKNOWN AGENT IDs:")
print(unknown_agents)

print("\nTRANSACTIONS WITH UNKNOWN AGENTS:")
print(txns["agent_id"].isin(unknown_agents).sum())

# 3. Check unusually large amounts
print("\nTRANSACTIONS WITH AMOUNT 250000:")
display(txns[txns["amount"] == 250000])

# 4. Examine duplicate transaction IDs
print("\nDUPLICATE TRANSACTION RECORDS:")
display(
    txns[txns.duplicated("txn_id", keep=False)]
    .sort_values("txn_id")
)

**Reusable cleaning module**

The cleaning function is maintained in the separate `cleaning_pipeline.py` file. This notebook imports that module below rather than rewriting or overwriting it.

In [ ]:
from cleaning_pipeline import clean_agent_data

clean_df, audit_log = clean_agent_data(txns, agents)

print("CLEANED DATA:")
print(clean_df.shape)
display(clean_df.head())

print("\nAUDIT LOG:")
display(audit_log)

In [ ]:
print("CLEANED DATA VALIDATION")
print("-----------------------")

print("Shape:", clean_df.shape)
print("Unique transaction IDs:", clean_df["txn_id"].nunique())
print("Duplicate transaction IDs:", clean_df["txn_id"].duplicated().sum())

print("\nTransaction types:")
print(clean_df["txn_type"].value_counts())

print("\nMissing values:")
print(clean_df.isna().sum())

print("\nAmount data type:", clean_df["amount"].dtype)
print("Date format examples:")
print(clean_df["txn_date"].head())

print("\nImputed amounts:", clean_df["amount_imputed"].sum())
print("Flagged outliers:", clean_df["is_outlier"].sum())
print("Unknown agents:", (clean_df["county"] == "Unknown").sum())

print("\nAUDIT LOG:")
display(audit_log)

In [ ]:
county_summary = clean_df.groupby(
    ["county", "txn_type"]
).agg(
    transaction_count=("txn_id", "count"),
    total_amount=("amount", "sum"),
    average_amount=("amount", "mean")
).reset_index()

display(county_summary)

In [ ]:
clean_df["txn_date"] = pd.to_datetime(clean_df["txn_date"])

clean_df["week"] = clean_df["txn_date"].dt.to_period("W").astype(str)

weekly_pivot = clean_df.pivot_table(
    index="county",
    columns="week",
    values="amount",
    aggfunc="sum",
    fill_value=0
)

display(weekly_pivot)

In [ ]:
import os

os.makedirs("outputs", exist_ok=True)

clean_df.to_csv(
    "outputs/agent_transactions_clean.csv",
    index=False
)

print("File saved successfully!")
print("Shape:", clean_df.shape)
print("File exists:", os.path.exists(
    "outputs/agent_transactions_clean.csv"
))

# Data Quality Report

## 1. Completeness
The original dataset contained 20 missing transaction amounts after duplicate removal. These values were replaced with the median amount, and all missing values were resolved.

## 2. Uniqueness
The dataset contained 12 duplicate transaction IDs. These were removed by retaining the first occurrence of each transaction ID. The cleaned dataset contains 600 unique transactions.

## 3. Consistency
A total of 376 transaction types were standardized to either `Deposit` or `Withdrawal`. Five transaction dates were converted to the ISO format (`YYYY-MM-DD`).

## 4. Validity
Eight transactions were associated with an unregistered agent ID (`AG099`). These records were retained and their county was labelled `Unknown`. No invalid transaction dates were found.

## 5. Accuracy
The IQR method identified 40 unusually high or low transaction amounts. These values were flagged for further investigation rather than deleted. Outliers are not necessarily errors.

## 6. Timeliness
The transaction records cover March and April 2026. Dates were standardized to support consistent chronological and weekly analysis.

## Data Privacy and Protection
Customer phone numbers are personal data under Kenya's Data Protection Act, 2019. If collected, they should be protected through encryption, restricted access, data minimization, secure storage and appropriate retention policies. Reports should use aggregated or anonymized information wherever possible.